In [ ]:
!pip install python-dotenv PyMuPDF google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 26.7 MB/s eta 0:00:00


In [1]:
# Install dependencies in Colab
!pip install PyMuPDF google-generativeai

#Import Packages
import os
import pathlib
import json
import fitz  # PyMuPDF
import google.generativeai as genai
from google.colab import files

def configure_llm():
    """
    Configures and returns the Generative AI model.
    Exits if the API key is not set.
    """
    api_key = os.environ.get("YOUR API KEY")
    genai.configure(api_key=api_key)
    return genai.GenerativeModel('gemini-2.0-flash')

def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extracts all text from a given PDF file.
    """
    if not pathlib.Path(pdf_path).is_file():
        print(f"Error: File not found at '{pdf_path}'")
        return ""

    try:
        doc = fitz.open(pdf_path)
        full_text = ""
        for page in doc:
            full_text += page.get_text()
        doc.close()
        print(f"✅ Successfully extracted text from '{pdf_path}'.")
        return full_text
    except Exception as e:
        print(f"Error processing PDF file: {e}")
        return ""

def generate_feature_descriptions(document_text: str, feature_list: list[str]) -> dict:
    """
    Uses an LLM to generate detailed descriptions for a list of features
    based on the provided document text.
    """
    model = configure_llm()

    prompt = f"""
    You are a professional QA analyst. Your task is to create detailed descriptions of software features based on a Functional Specification Document (FSD). These descriptions will be used to write comprehensive test cases.

    **Instructions:**
    1. Read the entire `DOCUMENT_TEXT` provided below.
    2. For each feature name in the `FEATURE_LIST`, locate the corresponding section in the `DOCUMENT_TEXT`.
    3. Synthesize a detailed, paragraph-style description for each feature. This description must include:
        - The primary goal or purpose of the feature. Also known as the process description for each feature.
        - The event/trigger for each feature.
        - The main user/actor who interacts with it.
        - The basic workflow or sequence of actions the user performs.
        - The alternate flow or sequence of actions the user performs.
        - The pre-condition
        - The post condition
        - The Business Rules, validations, or constraints mentioned.
    4. IMPORTANT: Use ONLY the information provided in the `DOCUMENT_TEXT`. Do not add any information or make assumptions.
    5. Format your final output as a single JSON object, where the keys are the exact feature names from the `FEATURE_LIST` and the values are the detailed string descriptions.

    **FEATURE_LIST:**
    {json.dumps(feature_list, indent=2)}

    **DOCUMENT_TEXT:**
    ---
    {document_text}
    ---

    **OUTPUT (JSON Object Only):**
    """

    print("\n🚀 Sending request to the LLM to generate descriptions. This may take a moment...")

    try:
        response = model.generate_content(prompt)
        cleaned_response = response.text.strip().replace("```json", "").replace("```", "")
        feature_descriptions = json.loads(cleaned_response)
        print("✅ Successfully received and parsed feature descriptions from the LLM.")
        return feature_descriptions
    except Exception as e:
        print(f"An error occurred while communicating with the LLM or parsing its response: {e}")
        print("\n--- Raw LLM Response ---")
        print(response.text if 'response' in locals() else "No response received.")
        print("------------------------")
        return {}

if __name__ == "__main__":
    # 1. Set your API key directly in Colab
    os.environ["GOOGLE_API_KEY"] = "YOUR API KEY"

    # 2. Upload your PDF file(s) via file picker
    uploaded = files.upload()
    pdf_document_path = list(uploaded.keys())[0]

    # 3. List features you want descriptions for
    features_to_describe = [
        # Here you can add all the features from any functional specification document that you want to generate description of.
    ]

    # 4. Extract text and generate descriptions
    fsd_text = extract_text_from_pdf(pdf_document_path)

    if fsd_text:
        descriptions = generate_feature_descriptions(fsd_text, features_to_describe)

        if descriptions:
            print("\n--- Generated Feature Descriptions ---")
            for feature, desc in descriptions.items():
                print(f"## Feature: {feature}\n")
                print(f"{desc}\n")
                print("-" * 40)
